<a href="https://colab.research.google.com/github/lc20140856-stack/Analisis-y-visualizacion-de-datos/blob/main/practica_bi_datamexEJO_ipyn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
install.packages("devtools")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [6]:
library(dplyr)
library(conflicted)
conflicts_prefer(dplyr::filter)

[conflicted] Will prefer dplyr::filter over any other package.


In [7]:
# R Script
## ── SECCIÓN I: PIPELINE ETL

## Instala los paquetes si no los tienes:
install.packages(c("tidyverse", "lubridate", "writexl", "scales", "DBI"))
library(tidyverse)
library(lubridate)
library(scales)
library(writexl)
## ── 1. EXTRACT: Generar dataset con errores intencionales ──────────────
set.seed(42)
n <- 500
ventas_raw <- tibble(
    id_venta = 1:n,
    fecha_venta = sample(c(
        format(seq(as.Date("2024-01-01"), as.Date("2024-12-31"), by="day"),
               c("%d/%m/%Y", "%Y-%m-%d")[sample(1:2, 366, replace=TRUE)]),
        NA_character_), n, replace=TRUE),
    id_producto = sample(paste0("P", sprintf("%03d", 1:30)), n,
                         replace=TRUE),
    producto = sample(c("Laptop", "LAPTOP", "laptop",
                        "Silla", "SILLA",
                        "Playera", "playera",
                        "Cereal", "CEREAL",
                        "Televisor", "televisor"), n, replace=TRUE),
    categoria = sample(c("Electrónico", "Hogar", "Ropa","Alimento", NA_character_), n,
                       replace=TRUE, prob=c(.25,.25,.2,.2,.1)),
    region = sample(c("Norte","Sur","Centro",
                      "Occidente","Oriente"), n, replace=TRUE),
    vendedor = sample(c("Ana López","ana lopez","ANA LOPEZ","Carlos Ruiz","carlos ruiz","María Torres",
                        "Javier Soto"), n,replace=TRUE),
    cantidad = sample(c(1:50, -5, -10, 0), n, replace=TRUE),
    precio_unitario = sample(c(100, 250, 500, 1200, 3500, 8000,
                               "$1500", "$250", NA_character_), n,
                             replace=TRUE),
    descuento_pct = sample(c(0,5,10,15,20,25,30,NA_real_), n,
                           replace=TRUE),
    canal_venta = sample(c("Online","Tienda","Distribuidor",
                           NA_character_), n, replace=TRUE)
    )
## ── Guardar como archivo de origen (simula el Extract) ────────────────
write_csv(ventas_raw, "ventas_raw_2024.csv")
cat("✅ EXTRACT completado:", nrow(ventas_raw), "registros extraídos\n")

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



✅ EXTRACT completado: 500 registros extraídos


In [9]:
# R Script
## ── 1.2 Diagnóstico inicial (Profile) ─────────────────────────────────
glimpse(ventas_raw)
summary(ventas_raw)
# Conteo de valores nulos por columna
ventas_raw %>%
summarise(across(everything(), ~sum(is.na(.)))) %>%
pivot_longer(everything(), names_to="columna", values_to="nulos") %>%
arrange(desc(nulos)) %>%
print()

Rows: 500
Columns: 11
$ id_venta        <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16,…
$ fecha_venta     <chr> "01/08/2024", "10/11/2024", "2024-04-22", "30/01/2024"…
$ id_producto     <chr> "P027", "P008", "P012", "P012", "P012", "P015", "P009"…
$ producto        <chr> "Televisor", "SILLA", "Cereal", "Televisor", "Playera"…
$ categoria       <chr> "Hogar", NA, NA, "Hogar", "Alimento", NA, "Ropa", "Ali…
$ region          <chr> "Sur", "Sur", "Norte", "Oriente", "Centro", "Sur", "Or…
$ vendedor        <chr> "Carlos Ruiz", "ANA LOPEZ", "Ana López", "ana lopez", …
$ cantidad        <dbl> 46, 29, 25, 14, 13, 1, 11, 35, 14, 24, 42, 43, 26, 37,…
$ precio_unitario <chr> "250", "8000", NA, "8000", "$250", NA, "1200", "1200",…
$ descuento_pct   <dbl> 25, 5, 30, NA, NA, 5, 20, 0, 5, 15, 10, NA, NA, 25, NA…
$ canal_venta     <chr> "Distribuidor", "Distribuidor", "Distribuidor", "Tiend…


    id_venta     fecha_venta        id_producto          producto        
 Min.   :  1.0   Length:500         Length:500         Length:500        
 1st Qu.:125.8   Class :character   Class :character   Class :character  
 Median :250.5   Mode  :character   Mode  :character   Mode  :character  
 Mean   :250.5                                                           
 3rd Qu.:375.2                                                           
 Max.   :500.0                                                           
                                                                         
  categoria            region            vendedor            cantidad     
 Length:500         Length:500         Length:500         Min.   :-10.00  
 Class :character   Class :character   Class :character   1st Qu.: 12.00  
 Mode  :character   Mode  :character   Mode  :character   Median : 25.00  
                                                          Mean   : 24.42  
                                 

# A tibble: 11 × 2
   columna         nulos
   <chr>           <int>
 1 canal_venta       120
 2 descuento_pct      67
 3 precio_unitario    54
 4 categoria          50
 5 fecha_venta         1
 6 id_venta            0
 7 id_producto         0
 8 producto            0
 9 region              0
10 vendedor            0
11 cantidad            0


In [8]:
dim(ventas_raw)

[1] 500  11

In [ ]:
# R Script
## ── 1.2 Diagnóstico inicial (Profile) ─────────────────────────────────
glimpse(ventas_raw)
summary(ventas_raw)
# Conteo de valores nulos por columna
ventas_raw %>%
summarise(across(everything(), ~sum(is.na(.)))) %>%
pivot_longer(everything(), names_to="columna", values_to="nulos") %>%
arrange(desc(nulos)) %>%
print()